## Logistics Delay Prediction – Modeling

### Objective
Predict the probability of delivery delays using the Olist dataset.
The target variable is `is_late`, where 1 indicates a delayed order.
Models return probabilities via `predict_proba()` rather than binary outputs.

### Plan

| Step | Description |
|---|---|
| 1 | Categorical encoding |
| 2 | Train/test split (stratified, 80/20) |
| 3 | Baseline – Dummy Classifier |
| 4 | Model training – LightGBM |
| 5 | Model training – CatBoost |
| 6 | Evaluation – ROC-AUC, F1, Recall |
| 7 | Hyperparameter tuning – Optuna |
| 8 | Interpretability – SHAP |

### Metrics
Given the class imbalance (~8% late orders), the following metrics are prioritized:
- ROC-AUC
- F1-Score
- Recall

### Stack
- scikit-learn
- LightGBM
- CatBoost
- Optuna
- SHAP

In [54]:
# ── Standard library ──────────────────────────────────────────────────────────
import warnings
warnings.filterwarnings("ignore")

# ── Data manipulation ─────────────────────────────────────────────────────────
import pandas as pd
import numpy as np

# ── Machine learning ──────────────────────────────────────────────────────────
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score, recall_score, classification_report
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import OrdinalEncoder

import lightgbm as lgb
from catboost import CatBoostClassifier

# ── Hyperparameter tuning ─────────────────────────────────────────────────────
import optuna
from optuna.samplers import TPESampler

# ── Interpretability ──────────────────────────────────────────────────────────
import shap

# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_parquet("consolidated_logistics_base.parquet")
print(f"Dataset loaded: {df.shape}")

Dataset loaded: (110196, 67)


In [55]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 67 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   seller_id                    110196 non-null  str    
 1   customer_state               110196 non-null  str    
 2   review_score                 109369 non-null  float64
 3   product_category_name        108659 non-null  str    
 4   product_weight_g             110178 non-null  float64
 5   payment_type_main            110193 non-null  str    
 6   seller_geo_city              109947 non-null  str    
 7   seller_geo_state             109947 non-null  str    
 8   customer_geo_city            109908 non-null  str    
 9   customer_geo_state           109908 non-null  str    
 10  seller_customer_distance_km  109660 non-null  float64
 11  is_late                      110196 non-null  Int8   
 12  product_volume_cm3           110178 non-null  float64
 13  is_heavy_p

In [56]:
df.head()

,seller_id,customer_state,review_score,product_category_name,product_weight_g,payment_type_main,seller_geo_city,seller_geo_state,customer_geo_city,customer_geo_state,...,bert_svd_41,bert_svd_42,bert_svd_43,bert_svd_44,bert_svd_45,bert_svd_46,bert_svd_47,bert_svd_48,bert_svd_49,estimated_delivery_dow
0,3504c0cb71d7fa48d967e0e4c94d59d9,SP,4.0,utilidades_domesticas,500.0,voucher,maua,SP,sao paulo,SP,...,-0.098109,0.146767,0.090446,0.280138,-0.106301,-0.437916,0.755499,0.160171,-0.121071,2
1,289cdb325fb7e7f891c38608bf9e0962,BA,4.0,perfumaria,400.0,boleto,belo horizonte,MG,barreiras,BA,...,0.058410,-0.147710,-0.116853,-0.269182,-0.241428,-0.200264,0.136254,-0.166536,0.043802,0
2,4869f7a5dfa277a7dca6462dcf3b52b2,GO,5.0,automotivo,420.0,credit_card,guariba,SP,vianopolis,GO,...,-0.001168,0.001993,0.000650,0.000121,-0.002988,0.000843,-0.000631,-0.002070,-0.001148,1
3,66922902710d126a0e7d26b0e3805106,RN,5.0,pet_shop,450.0,credit_card,belo horizonte,MG,sao goncalo do amarante,RN,...,0.052156,-0.085034,0.380572,0.057153,0.047215,0.005328,-0.367307,-0.231086,0.072944,4
4,2c9e548be18521d1c43cde1c582c6de8,SP,5.0,papelaria,250.0,credit_card,mogi das cruzes,SP,santo andre,SP,...,-0.001168,0.001993,0.000650,0.000121,-0.002988,0.000843,-0.000631,-0.002070,-0.001148,0


In [57]:
# Check target distribution
df["is_late"].value_counts(normalize=True).round(3)

is_late
0    0.921
1    0.079
Name: proportion, dtype: Float64

## Initial Inspection – Main Findings

- Dataset loaded with **110,196 rows** and **67 columns**, consuming ~41 MB in memory.
- Target variable `is_late` is highly imbalanced: **92% on time vs. 8% late** — requires `class_weight='balanced'` or `scale_pos_weight` during training.
- **8 categorical columns** (dtype `str`) require encoding before modeling: `customer_state`, `seller_geo_state`, `customer_geo_state`, `seller_geo_city`, `customer_geo_city`, `product_category_name`, `payment_type_main`, and `seller_id`.
- **50 BERT-SVD features** (float32) are already numeric and ready for modeling.
- A small number of nulls were identified in `review_score`, `product_category_name`, `product_weight_g`, and geo columns — to be handled before split.
- No duplicate columns or unexpected dtypes detected.

In [58]:
categorical_cols = [
    "customer_state", "seller_geo_state", "customer_geo_state",
    "seller_geo_city", "customer_geo_city", "product_category_name",
    "payment_type_main"
]

df = df.drop(columns=["seller_id"])

encoder = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
df[categorical_cols] = encoder.fit_transform(df[categorical_cols])

In [59]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 110196 entries, 0 to 110195
Data columns (total 66 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   customer_state               110196 non-null  float64
 1   review_score                 109369 non-null  float64
 2   product_category_name        108659 non-null  float64
 3   product_weight_g             110178 non-null  float64
 4   payment_type_main            110193 non-null  float64
 5   seller_geo_city              109947 non-null  float64
 6   seller_geo_state             109947 non-null  float64
 7   customer_geo_city            109908 non-null  float64
 8   customer_geo_state           109908 non-null  float64
 9   seller_customer_distance_km  109660 non-null  float64
 10  is_late                      110196 non-null  Int8   
 11  product_volume_cm3           110178 non-null  float64
 12  is_heavy_product             110196 non-null  Int8   
 13  is_bulky_p

In [60]:
(df["customer_state"] == df["customer_geo_state"]).mean()

np.float64(0.9973864750081672)

## Initial Inspection – Main Findings

- Dataset loaded with **110,196 rows** and **67 columns**, consuming ~41 MB in memory.
- Target variable `is_late` is highly imbalanced: **92% on time vs. 8% late** — requires `class_weight='balanced'` or `scale_pos_weight` during training.
- `customer_geo_state` and `seller_geo_state` dropped — 99.7% identical to `customer_state` and `seller_geo_state` respectively.
- `seller_id` dropped for modeling — high cardinality; reserved for Power BI dashboard.
- **50 BERT-SVD features** (float32) already numeric and ready for modeling.
- Small number of nulls identified in `review_score`, `product_category_name`, `product_weight_g` and geo columns — to be handled before split.

## Encoding Strategy

| Column | Encoding | Rationale |
|---|---|---|
| customer_state | One-Hot | Low cardinality (27 states). No implicit order. |
| seller_geo_city | Target Encoding (k-fold) | High cardinality. Captures predictive signal without dimensionality explosion. |
| customer_geo_city | Target Encoding (k-fold) | Same as above. |
| product_category_name | Target Encoding (k-fold) | Medium-high cardinality (~70 categories). More robust than one-hot. |
| payment_type_main | One-Hot | Very low cardinality (4–5 types). No implicit order. |

In [61]:
# ── Drop redundant and high-cardinality columns ───────────────────────────────
df = df.drop(columns=["customer_geo_state", "seller_geo_state"])

# ── One-Hot Encoding ──────────────────────────────────────────────────────────
ohe_cols = ["customer_state", "payment_type_main"]
df = pd.get_dummies(df, columns=ohe_cols, drop_first=False, dtype=np.int8)

# ── Target Encoding (k-fold) ──────────────────────────────────────────────────
target_enc_cols = ["seller_geo_city", "customer_geo_city", "product_category_name"]
target = df["is_late"].astype(float)
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for col in target_enc_cols:
    df[col + "_te"] = np.nan
    for train_idx, val_idx in kf.split(df):
        mean_enc = target.iloc[train_idx].groupby(df[col].iloc[train_idx]).mean()
        df.loc[df.index[val_idx], col + "_te"] = df[col].iloc[val_idx].map(mean_enc)
    global_mean = target.mean()
    df[col + "_te"] = df[col + "_te"].fillna(global_mean)
    df = df.drop(columns=[col])

print(df.shape)
df.head(3)

(110196, 93)


,review_score,product_weight_g,seller_customer_distance_km,is_late,product_volume_cm3,is_heavy_product,is_bulky_product,has_comment,bert_svd_0,bert_svd_1,...,customer_state_24.0,customer_state_25.0,customer_state_26.0,payment_type_main_0.0,payment_type_main_1.0,payment_type_main_2.0,payment_type_main_3.0,seller_geo_city_te,customer_geo_city_te,product_category_name_te
0,4.0,500.0,18.566632,0,1976.0,0,0,1,-8.512578,-5.002414,...,0,1,0,0,0,0,1,0.020661,0.062273,0.065185
1,4.0,400.0,847.437333,0,4693.0,0,0,1,-8.178789,-4.592004,...,0,0,0,1,0,0,0,0.046867,0.068182,0.076723
2,5.0,420.0,512.100044,0,9576.0,0,0,0,-9.135558,2.350698,...,0,0,0,0,1,0,0,0.116558,0.000000,0.085460


In [62]:
X = df.drop(columns=["is_late"])
y = df["is_late"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train distribution:\n{y_train.value_counts(normalize=True).round(3)}")
print(f"y_test distribution:\n{y_test.value_counts(normalize=True).round(3)}")

X_train: (88156, 92) | X_test: (22040, 92)
y_train distribution:
is_late
0    0.921
1    0.079
Name: proportion, dtype: float64
y_test distribution:
is_late
0    0.921
1    0.079
Name: proportion, dtype: float64


## Train/Test Split – Main Findings

- Split applied with 80/20 ratio: 88,156 rows for training and 22,040 for testing.
- `stratify=y` ensures class balance is preserved in both sets: 92.1% on time vs. 7.9% late.
- Final feature set: 92 columns after encoding.
- Target variable: `is_late` (int), binary classification.

In [63]:
dummy = DummyClassifier(strategy="stratified", random_state=42)
dummy.fit(X_train, y_train)

y_prob_dummy = dummy.predict_proba(X_test)[:, 1]
y_pred_dummy = dummy.predict(X_test)

print(classification_report(y_test, y_pred_dummy))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_dummy):.4f}")

              precision    recall  f1-score   support

           0       0.92      0.92      0.92     20297
           1       0.09      0.09      0.09      1743

    accuracy                           0.86     22040
   macro avg       0.51      0.51      0.51     22040
weighted avg       0.86      0.86      0.86     22040

ROC-AUC: 0.5078


## Baseline – DummyClassifier

- Strategy: `stratified` — predicts randomly respecting the class distribution (92% / 8%).
- ROC-AUC: 0.5078 (near random chance).
- Recall for class 1 (late): 0.09 — minimal delay detection, driven purely by chance.
- Accuracy: 0.86 — misleading metric given class imbalance; ignored in favor of ROC-AUC and Recall.
- This result sets the minimum performance bar for subsequent models.

In [64]:
lgbm = lgb.LGBMClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

lgbm.fit(X_train, y_train)

y_prob_lgbm = lgbm.predict_proba(X_test)[:, 1]
y_pred_lgbm = lgbm.predict(X_test)

print(classification_report(y_test, y_pred_lgbm))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lgbm):.4f}")

[LightGBM] [Info] Number of positive: 6972, number of negative: 81184
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.023901 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 14307
[LightGBM] [Info] Number of data points in the train set: 88156, number of used features: 92
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
              precision    recall  f1-score   support

           0       0.97      0.88      0.92     20297
           1       0.32      0.67      0.44      1743

    accuracy                           0.86     22040
   macro avg       0.65      0.78      0.68     22040
weighted avg       0.92      0.86      0.88     22040

ROC-AUC: 0.8348


## LightGBM – Default Parameters

- ROC-AUC: 0.8348
- Recall (class 1): 0.67 — detects 67% of actual delays.
- Precision (class 1): 0.32 — expected trade-off given class imbalance (~8% late).
- F1 (class 1): 0.44
- No hyperparameter tuning applied. `class_weight="balanced"` used to handle imbalance.

In [65]:
catboost = CatBoostClassifier(
    auto_class_weights="Balanced",
    random_seed=42,
    verbose=0
)

catboost.fit(X_train, y_train)

y_prob_cat = catboost.predict_proba(X_test)[:, 1]
y_pred_cat = catboost.predict(X_test)

print(classification_report(y_test, y_pred_cat))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_cat):.4f}")

              precision    recall  f1-score   support

           0       0.97      0.90      0.93     20297
           1       0.37      0.65      0.47      1743

    accuracy                           0.88     22040
   macro avg       0.67      0.78      0.70     22040
weighted avg       0.92      0.88      0.90     22040

ROC-AUC: 0.8323


## CatBoost – Default Parameters

- ROC-AUC: 0.8323
- Recall (class 1): 0.65 — detects 65% of actual delays.
- Precision (class 1): 0.37 — slightly higher than LightGBM, fewer false alarms.
- F1 (class 1): 0.47
- No hyperparameter tuning applied. `auto_class_weights="Balanced"` used to handle imbalance.

In [66]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective(trial):
    """
    Optuna objective function for LightGBM hyperparameter tuning.

    Defines the search space for key LightGBM parameters and evaluates
    each trial using 5-fold stratified cross-validation on the training set.
    Returns the mean ROC-AUC across folds, which Optuna uses to guide
    the search toward better hyperparameter regions.

    Parameters
    ----------
    trial : optuna.trial.Trial
        A single optimization trial managed by Optuna.

    Returns
    -------
    float
        Mean ROC-AUC score across the 5 stratified folds.
    """
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.3, log=True),
        "num_leaves": trial.suggest_int("num_leaves", 20, 300),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
        "class_weight": "balanced",
        "random_state": 42,
        "n_jobs": -1,
        "verbose": -1,
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, val_idx in cv.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = lgb.LGBMClassifier(**params)
        model.fit(X_tr, y_tr)

        # Use predicted probabilities for class 1 to compute ROC-AUC
        y_prob = model.predict_proba(X_val)[:, 1]
        scores.append(roc_auc_score(y_val, y_prob))

    return np.mean(scores)

study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f"Best ROC-AUC: {study.best_value:.4f}")
print(f"Best params:  {study.best_params}")

Best trial: 36. Best value: 0.829121: 100%|██████████| 50/50 [1:12:12<00:00, 86.65s/it] 

Best ROC-AUC: 0.8291
Best params:  {'n_estimators': 806, 'learning_rate': 0.007067705547716829, 'num_leaves': 179, 'max_depth': 8, 'min_child_samples': 92, 'subsample': 0.9175155346892794, 'colsample_bytree': 0.5021345958158536, 'reg_alpha': 0.00010637374896307073, 'reg_lambda': 9.952257778061577}


In [67]:
# ── LightGBM – Tuned Model ────────────────────────────────────────────────────
best_params = study.best_params

lgbm_tuned = lgb.LGBMClassifier(
    **best_params,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgbm_tuned.fit(X_train, y_train)

y_prob_lgbm_tuned = lgbm_tuned.predict_proba(X_test)[:, 1]
y_pred_lgbm_tuned = lgbm_tuned.predict(X_test)

print(classification_report(y_test, y_pred_lgbm_tuned))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_lgbm_tuned):.4f}")

              precision    recall  f1-score   support

           0       0.97      0.89      0.93     20297
           1       0.34      0.66      0.45      1743

    accuracy                           0.87     22040
   macro avg       0.66      0.77      0.69     22040
weighted avg       0.92      0.87      0.89     22040

ROC-AUC: 0.8395


In [68]:
# ── Feature Importance – LightGBM Tuned ───────────────────────────────────────
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": lgbm_tuned.feature_importances_
}).sort_values("importance", ascending=False)

# Show SVD columns specifically
svd_cols = feature_importance[feature_importance["feature"].str.contains("svd")]
print(svd_cols.to_string(index=False))

    feature  importance
 bert_svd_4        2409
bert_svd_18        1896
 bert_svd_7        1893
bert_svd_14        1858
 bert_svd_8        1830
 bert_svd_2        1792
bert_svd_10        1790
bert_svd_25        1690
bert_svd_11        1686
 bert_svd_3        1632
bert_svd_44        1628
bert_svd_38        1526
bert_svd_16        1520
 bert_svd_6        1512
bert_svd_30        1498
bert_svd_27        1488
bert_svd_24        1427
 bert_svd_9        1407
bert_svd_41        1344
bert_svd_21        1340
bert_svd_15        1315
 bert_svd_5        1313
bert_svd_32        1292
bert_svd_13        1261
bert_svd_22        1233
bert_svd_43        1179
bert_svd_28        1172
bert_svd_34        1131
bert_svd_48        1113
 bert_svd_0        1101
bert_svd_19        1095
 bert_svd_1        1093
bert_svd_29        1093
bert_svd_40        1079
bert_svd_46        1077
bert_svd_26        1067
bert_svd_47        1061
bert_svd_23        1044
bert_svd_20        1043
bert_svd_36        1036
bert_svd_17     

In [69]:
# ── LightGBM – No SVD Features ────────────────────────────────────────────────
svd_cols = [col for col in X_train.columns if col.startswith("bert_svd")]

X_train_no_svd = X_train.drop(columns=svd_cols)
X_test_no_svd = X_test.drop(columns=svd_cols)

lgbm_no_svd = lgb.LGBMClassifier(
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

lgbm_no_svd.fit(X_train_no_svd, y_train)

y_prob_no_svd = lgbm_no_svd.predict_proba(X_test_no_svd)[:, 1]
y_pred_no_svd = lgbm_no_svd.predict(X_test_no_svd)

print(classification_report(y_test, y_pred_no_svd))
print(f"ROC-AUC: {roc_auc_score(y_test, y_prob_no_svd):.4f}")

              precision    recall  f1-score   support

           0       0.97      0.83      0.89     20297
           1       0.25      0.66      0.36      1743

    accuracy                           0.82     22040
   macro avg       0.61      0.75      0.63     22040
weighted avg       0.91      0.82      0.85     22040

ROC-AUC: 0.8154


## Feature Evaluation – SVD Components

### Context

The model showed low precision for class 1 (delayed orders), raising concerns about
the cost of false positives in a logistics context. Before adjusting the decision
threshold, we evaluated whether the 50 BERT SVD components were contributing to
model performance or adding noise.

A secondary concern was potential data leakage: review-based features are filled
by the customer after delivery, meaning they would not be available at prediction time
in a real production scenario.

### Experiment

We trained a LightGBM default model (class_weight="balanced") on two feature sets:
- Full feature set (including bert_svd_0 to bert_svd_49)
- Feature set excluding all SVD columns (50 columns dropped)

### Results

| Metric            | LightGBM (full) | LightGBM (no SVD) |
|-------------------|-----------------|-------------------|
| ROC-AUC           | 0.8348          | 0.8154            |
| Recall (class 1)  | 0.67            | 0.66              |
| Precision (class 1)| 0.32           | 0.25              |
| F1 (class 1)      | 0.44            | 0.36              |

### Conclusion

SVD components are contributing meaningfully to model performance. Dropping them
caused a notable drop in ROC-AUC and precision. Despite the data leakage concern,
they will be retained for now as this is a portfolio project. This trade-off should
be clearly documented if the model is presented in a production context.

The low precision issue remains open and will be addressed via decision threshold
tuning in the next step.